# Calling `quantem-cuda` kernels directly

[`quantem-cuda`](https://github.com/electronmicroscopy/quantem-cuda) is an optional companion
package of hand-written CUDA C++ kernels (with analytic gradients) for quantem's compute
hot spots. quantem dispatches to it transparently when it is installed (see the last
section), but every kernel is also a plain public function you can call from any torch
code — they take and return `torch.Tensor`, are registered as torch custom ops, and
compose with autograd and `torch.compile` without graph breaks.

This notebook shows the direct-call API:

1. **Total-variation losses** (`quantem.cuda.core`) — *exactly* what is and isn't
   implemented, with parity checks against pure-torch references.
2. **Using the TV kernels in ptychography workflows** — complex multislice objects,
   per-axis weighting, and what the switch means for your regularizer.
3. **The fused TILTED K-Planes interpolation** (`quantem.cuda.core.ml`).
4. **Transparent dispatch inside quantem** and the kill switch.

**Install** (CUDA 13 toolchain wheels at build time; no system CUDA needed):

```bash
pip install 'nvidia-cuda-nvcc==13.0.*' 'nvidia-cuda-cccl==13.0.*' \
            'nvidia-cuda-runtime==13.0.*' 'nvidia-cuda-crt==13.0.*' 'nvidia-nvvm==13.0.*'
pip install git+https://github.com/electronmicroscopy/quantem-cuda   # PyPI: quantem[cuda], once published
```


In [1]:
import torch

import quantem.cuda
from quantem.cuda.core import tv_loss_iso_3d, tv_loss_sq_3d
from quantem.cuda.core.ml import kplanes_tilted_fuse

print("torch        :", torch.__version__)
print("device       :", torch.cuda.get_device_name(0))
print("quantem-cuda :", quantem.cuda.__version__, "| cudart", quantem.cuda.cudart_version())

torch        : 2.12.0+cu130
device       : NVIDIA RTX PRO 6000 Blackwell Server Edition
quantem-cuda : 0.1.0 | cudart 13000


## 1. The TV-loss kernels — exactly what is implemented

`quantem.cuda.core` ships **two** total-variation functionals, each as a fused forward
kernel plus a fused analytic-backward kernel (one launch each, no large intermediates):

### `tv_loss_sq_3d(volume)` — squared anisotropic TV (quantem `tv_vol` parity)

$$\mathcal{L} = \sum (\Delta_d v)^2 \; + \; \sum (\Delta_h v)^2 \; + \; \sum (\Delta_w v)^2$$

Forward differences along each of the three trailing dims, each squared and summed over
its full index range. **Returns the raw, unnormalized sum** — bit-for-bit the quantity
quantem's tomography `tv_vol` regularizer computes before its `weight / numel` scaling;
you apply your own weighting/normalization.

### `tv_loss_iso_3d(volume, eps=1e-8)` — isotropic (edge-preserving) TV

$$\mathcal{L} = \operatorname{mean}_{\text{corners}} \sqrt{(\Delta_d v)^2 + (\Delta_h v)^2 + (\Delta_w v)^2 + \varepsilon}$$

The three forward differences are evaluated on the common corner set
$[0,D\!-\!2]\times[0,H\!-\!2]\times[0,W\!-\!2]$ and the result is **mean-reduced over
that corner set** (leading dims included). $\varepsilon$ sits *inside* the square root.

### Shared contract (both kernels)

- input: **fp32, CUDA, real** tensor of shape `[D, H, W]` or `[..., D, H, W]` (`ndim >= 3`);
  any leading batch/channel dims are flattened and included in the reduction;
- returns a 0-dim fp32 tensor on the same device; fully differentiable (hand-written
  backward, gradients exact in one kernel launch); `torch.compile(fullgraph=True)`-safe;
- raises `TypeError`/`ValueError` on non-fp32, non-CUDA, or `ndim < 3` input — there is
  **no CPU fallback** in the package (quantem's dispatch layer handles the fallback).

### Explicitly **not** implemented (so you don't discover it the hard way)

- **Anisotropic L1 TV** — `mean(|diff|)` per axis. This is what
  `quantem.diffractive_imaging`'s `ObjectConstraints._calc_tv_loss` computes today
  (with separate `tv_weight_z` / `tv_weight_xy` weights). Calling a kernel instead
  **changes the regularizer functional**, not just its speed — see section 2.
- **Per-axis weights** inside a kernel — both kernels weight all three axes equally
  (a two-call recipe below recovers per-axis weighting for the squared variant).
- A 2-D-only variant. `tv_loss_sq_3d` on `[1, H, W]` degrades gracefully (the depth term
  is an empty sum, so you get pure-2D squared TV); `tv_loss_iso_3d` on `[1, H, W]` has an
  **empty corner set and returns 0.0** — it needs `D >= 2` to measure anything.
- Complex tensors (take `.angle()` / `.abs()` first), fp16 / bf16 / fp64, Huber/smoothed-L1
  variants, and any normalization other than stated above.

In [2]:
# Parity check: tv_loss_sq_3d == quantem's tv_vol functional (raw sum)
def torch_tv_sq(v):
    return (
        (v[..., 1:, :, :] - v[..., :-1, :, :]).pow(2).sum()
        + (v[..., :, 1:, :] - v[..., :, :-1, :]).pow(2).sum()
        + (v[..., :, :, 1:] - v[..., :, :, :-1]).pow(2).sum()
    )

vol = torch.rand(64, 128, 128, device="cuda", requires_grad=True)
ref, fused = torch_tv_sq(vol), tv_loss_sq_3d(vol)

(g_ref,) = torch.autograd.grad(ref, vol, retain_graph=True)
(g_fused,) = torch.autograd.grad(fused, vol)

print(f"value : torch {ref.item():.4f}  kernel {fused.item():.4f}")
print("grads :", torch.allclose(g_ref, g_fused, rtol=1e-4, atol=1e-5))

value : torch 518341.5312  kernel 518341.0625
grads : True


In [3]:
# Parity check: tv_loss_iso_3d == corner-restricted isotropic TV (mean-reduced)
def torch_tv_iso(v, eps=1e-8):
    # all three forward differences anchored at corner (i, j, k)
    dd = (v[..., 1:, :, :] - v[..., :-1, :, :])[..., :, :-1, :-1]
    dh = (v[..., :, 1:, :] - v[..., :, :-1, :])[..., :-1, :, :-1]
    dw = (v[..., :, :, 1:] - v[..., :, :, :-1])[..., :-1, :-1, :]
    return (dd.pow(2) + dh.pow(2) + dw.pow(2) + eps).sqrt().mean()

ref, fused = torch_tv_iso(vol), tv_loss_iso_3d(vol)
(g_ref,) = torch.autograd.grad(ref, vol, retain_graph=True)
(g_fused,) = torch.autograd.grad(fused, vol)

print(f"value : torch {ref.item():.6f}  kernel {fused.item():.6f}")
print(f"grads : {torch.allclose(g_ref, g_fused, rtol=1e-4, atol=1e-6)} (max abs diff {(g_ref - g_fused).abs().max().item():.1e})")
print("iso on a single slice (D=1) is identically zero:", tv_loss_iso_3d(torch.rand(1, 256, 256, device='cuda')).item())

value : torch 0.654233  kernel 0.654232
grads : True (max abs diff 9.1e-13)
iso on a single slice (D=1) is identically zero: 0.0


## 2. Using the TV kernels in ptychography workflows

quantem's ptychography object constraint (`ObjectConstraints.get_tv_loss`) applies
**anisotropic L1** TV — `mean(|diff|)` per axis, weighted by `(tv_weight_z, tv_weight_xy)` —
to the **phase** of the object (and to the amplitude too for `obj_type="complex"`).
Neither CUDA kernel computes that functional, so there are two distinct questions:

**(a) Same regularizer, faster?** Not available — there is no L1 kernel today. If your
reconstruction is tuned around L1-TV behavior (edge preservation, sparse gradients),
keep the torch path.

**(b) A different (but standard) TV flavor, much faster?** Both kernels apply directly to
the same tensors ptychography regularizes:

- the object is complex `(num_slices, H, W)` — call the kernel on `obj.angle()`
  (fp32, real), exactly the tensor the L1 path regularizes;
- `tv_loss_sq_3d` (quadratic smoothing — stronger penalty on large jumps, gentler on
  small texture) or `tv_loss_iso_3d` (the classic edge-preserving isotropic TV;
  needs `num_slices >= 2`);
- per-axis weighting for the squared variant via two calls (verified below):
  `tv_xy = tv_loss_sq_3d(obj_phase.unsqueeze(1))` (each slice as its own depth-1 volume →
  xy terms only) and `tv_z = tv_loss_sq_3d(obj_phase) - tv_xy`.

In [4]:
# Ptychography-shaped demo: complex multislice object, per-axis weighted squared TV
num_slices, H, W = 16, 512, 512
obj = torch.polar(
    torch.rand(num_slices, H, W, device="cuda") + 0.5,
    torch.rand(num_slices, H, W, device="cuda"),
).requires_grad_(True)  # complex64, like a multislice ptycho object

phase = obj.angle()  # fp32 real — same tensor the L1 constraint regularizes

w_z, w_xy = 5.0, 0.1  # e.g. the multislice tutorial's tv_weight_z-dominant setup
tv_xy = tv_loss_sq_3d(phase.unsqueeze(1))      # (S, 1, H, W): depth diffs are empty -> xy only
tv_z = tv_loss_sq_3d(phase) - tv_xy            # all-axis sum minus xy = z only
loss = (w_z * tv_z + w_xy * tv_xy) / phase.numel()
loss.backward()  # flows through .angle() back to the complex object

ref_z = (phase[1:] - phase[:-1]).pow(2).sum()
ref_xy = (phase[:, 1:, :] - phase[:, :-1, :]).pow(2).sum() + (phase[:, :, 1:] - phase[:, :, :-1]).pow(2).sum()
print("per-axis recipe matches torch:",
      torch.allclose(tv_z.detach(), ref_z, rtol=1e-3),
      torch.allclose(tv_xy.detach(), ref_xy, rtol=1e-4))
print("complex-object grad present  :", obj.grad is not None and bool(obj.grad.abs().sum() > 0))

per-axis recipe matches torch: True True
complex-object grad present  : True


In [5]:
# Op-level timings at ptychography- and tomography-representative sizes.
# 'L1 (torch)' is the functional ptychography uses today (no kernel exists for it);
# the kernel columns are the two implemented functionals, fwd+bwd, median of 30.
def _time(fn, x, iters=30):
    for _ in range(5):
        y = fn(x); y.backward(); x.grad = None
    s, e = torch.cuda.Event(True), torch.cuda.Event(True)
    ts = []
    for _ in range(iters):
        torch.cuda.synchronize(); s.record()
        y = fn(x); y.backward(); x.grad = None
        e.record(); torch.cuda.synchronize()
        ts.append(s.elapsed_time(e))
    return sorted(ts)[len(ts) // 2]

def torch_tv_l1(v):  # ptycho _calc_tv_loss, all axis weights equal
    return sum(torch.mean(torch.abs(v.diff(dim=d))) for d in range(v.ndim)) / v.ndim

print(f"{'shape':>17} | {'L1 (torch)':>10} | {'sq torch':>9} {'sq kernel':>9} | {'iso torch':>9} {'iso kernel':>10}")
for shape in [(1, 1024, 1024), (16, 512, 512), (16, 1024, 1024), (256, 256, 256)]:
    x = torch.rand(*shape, device="cuda", requires_grad=True)
    row = [_time(torch_tv_l1, x), _time(torch_tv_sq, x), _time(tv_loss_sq_3d, x)]
    if shape[0] > 1:
        row += [_time(torch_tv_iso, x), _time(tv_loss_iso_3d, x)]
        print(f"{str(shape):>17} | {row[0]:>8.2f}ms | {row[1]:>7.2f}ms {row[2]:>7.2f}ms | {row[3]:>7.2f}ms {row[4]:>8.2f}ms")
    else:
        print(f"{str(shape):>17} | {row[0]:>8.2f}ms | {row[1]:>7.2f}ms {row[2]:>7.2f}ms | {'(D=1: iso n/a)':>20}")

            shape | L1 (torch) |  sq torch sq kernel | iso torch iso kernel
  (1, 1024, 1024) |     0.27ms |    0.25ms    0.16ms |       (D=1: iso n/a)
   (16, 512, 512) |     0.34ms |    0.33ms    0.15ms |    0.45ms     0.18ms


 (16, 1024, 1024) |     2.17ms |    1.96ms    0.23ms |    3.23ms     0.36ms


  (256, 256, 256) |     2.19ms |    1.97ms    0.23ms |    3.34ms     0.37ms


**Is it tangible for a ptychography reconstruction?** The TV term runs once per iteration,
so compare the saving against your iteration time. On this GPU (RTX PRO 6000), switching a
`(16, 1024, 1024)` multislice phase from the torch L1 chain to the fused squared kernel
saves on the order of 2–5 ms per iteration (run-to-run clock variance on this shared
machine); for single-slice `(1, 1024, 1024)` objects the saving is ~0.1 ms — launch
overhead floors dominate at that size. If your iteration is dominated by the
per-probe-position FFT forward model (typically tens to hundreds of ms), expect a
few-percent end-to-end win at large multislice sizes and roughly nothing for
small/single-slice objects — the kernels exist because *tomography* volumes (256³–512³,
where the fused kernels are ~10× and ~30× faster respectively on an op that matters)
needed them. The honest summary: use the kernels for ptychography if (a) you want the
squared or isotropic functional anyway, or (b) your object is large multislice and you
are counting milliseconds; not as a drop-in L1 replacement.

## 3. The fused TILTED K-Planes interpolation (`quantem.cuda.core.ml`)

`kplanes_tilted_fuse(pts, rotations, plane)` fuses one multiscale level of quantem's
`interpolate_ms_features_tilted` (rotate → tri-plane bilinear sample → Hadamard product)
into a single kernel pair:

- `pts` — fp32 `[B, 3]`, coordinates in $[-1, 1]^3$ (border-clamped outside);
- `rotations` — fp32 `[T, 3, 3]` rotation matrices (the TILTED basis set);
- `plane` — fp32 `[3T, C, H, W]` plane grids, plane index `t*3 + {XY, ZX, YZ}`;
- returns `[B, T*C]` features; gradients flow to **all three** inputs (points → pose,
  rotations → SO(3) parameters, grids → the model).

Sampling semantics match `F.grid_sample(align_corners=True, padding_mode="border")`
exactly, gradients included. It lives in `core.ml` (not a per-technique module) because
`KPlanesTILTED` itself lives in `quantem.core.ml.models.kplanes` — any K-Planes-based
model (tomography object models today, other tensor-decomposition applications tomorrow)
goes through the same op.

In [6]:
from quantem.core import config
from quantem.core.ml.models.kplanes import interpolate_ms_features_tilted
from torch import nn

B, T, C, scales = 200_000, 4, 8, (64, 128)
pts = (torch.rand(B, 3, device="cuda") * 2 - 1).requires_grad_(True)
rot = torch.linalg.qr(torch.randn(T, 3, 3, device="cuda")).Q.contiguous().requires_grad_(True)
grids = nn.ParameterList(
    nn.Parameter(torch.rand(3 * T, C, s, s, device="cuda") * 0.4 + 0.1) for s in scales
)

# direct kernel call, one level at a time (this is all the fused op is):
feats_kernel = torch.cat([kplanes_tilted_fuse(pts, rot, g) for g in grids], dim=-1)

# pure-torch reference path (kill switch forces it):
config.set({"use_cuda_kernels": False})
try:
    feats_torch = interpolate_ms_features_tilted(pts, grids, rot)
finally:
    config.set({"use_cuda_kernels": True})

diff = (feats_kernel - feats_torch).abs().max().item()
print(f"features match torch chain: {torch.allclose(feats_kernel, feats_torch, rtol=1e-3, atol=1e-4)} (max abs diff {diff:.2e})")

def run_kernel(_):
    return torch.cat([kplanes_tilted_fuse(pts, rot, g) for g in grids], dim=-1).square().sum()
def run_torch(_):
    return interpolate_ms_features_tilted(pts, grids, rot).square().sum()

config.set({"use_cuda_kernels": False})
try:
    t_torch = _time(run_torch, pts)
finally:
    config.set({"use_cuda_kernels": True})
t_kernel = _time(run_kernel, pts)
print(f"fwd+bwd, B={B:,}, T={T}, C={C}, scales={scales}: torch {t_torch:.2f} ms | kernel {t_kernel:.2f} ms | {t_torch / t_kernel:.1f}x")

features match torch chain: True (max abs diff 1.64e-06)


fwd+bwd, B=200,000, T=4, C=8, scales=(64, 128): torch 4.37 ms | kernel 1.04 ms | 4.2x


## 4. Transparent dispatch inside quantem

With `quantem-cuda` installed you normally never call the kernels yourself:

- `config.get("has_quantem_cuda")` reports whether the package imported successfully;
- `interpolate_ms_features_tilted` (every `KPlanesTILTED` forward — data term, TV soft
  constraints, volume decoding) dispatches per multiscale level;
- `quantem.tomography.utils.tv_loss_vol_sq` (the `ObjectPixelated` TV regularizer)
  dispatches to `tv_loss_sq_3d`;
- dispatch engages only for fp32 CUDA tensors and falls back to the identical pure-torch
  code otherwise;
- `config.set({"use_cuda_kernels": False})` is the global kill switch (default `True`).

Ptychography's L1 TV constraint is **not** dispatched (different functional, see section 2);
calling the kernels directly, as above, is the supported way to use them there today.

In [7]:
print("has_quantem_cuda :", config.get("has_quantem_cuda"))
print("use_cuda_kernels :", config.get("use_cuda_kernels", default=True))

has_quantem_cuda : True
use_cuda_kernels : True
